# Oracle Agent Memory – Concepts and API Walkthrough

This notebook includes:
1. An overview of how memory is structured and managed within Oracle Agent Memory
2. An end-to-end example demonstrating how to use the OAM API, including setup instructions
3. Practical notes
This notebook does not cover agent creation. Instead, it focuses on using LLM models to explore and demonstrate OAM features.

Resource: 
1. API Documentation: https://docs.oracle.com/en/database/oracle/agent-memory/26.4/agmea/api/index.html#Agent-Memory
1. Example: https://docs.oracle.com/en/database/oracle/agent-memory/26.4/agmea/code-samples.html#GUID-BD5A7A3F-5F29-44DD-97B0-9AAE012C3391

Date: 04/05/2026 

Author: Assaf Rabinowicz, Data Scientist, EMEA

## Oracle Agent Memory - Core Concepts

Oracle Agent Memory (OAM) provides a persistent memory layer for AI agents, combining database storage, semantic search, and LLM processing to transform conversations into structured, reusable knowledge.

* The tool enables organizing and abstracting **user** profiles and interactions with LLMs.
* Raw **messages** between the user and the LLM are stored in a **thread**. An **agent** can also be associated with a thread.
* Messages are not only stored as raw data. The LLM processes them and extracts valuable structured information, such as **facts**, **preferences**, and **guidelines**.
* Users can also add **memory** manually and define ahead user profiles.
* The system uses embeddings and semantic **search** to retrieve relevant information.
* This structured memory enables effective agent interactions and personalization.

## Workspace Setup

Requirements
1. Oracle 26AI Database
2. Installation of the oracleagentmemory package
3. Access to at least one LLM model and one embedding model from the compatible model list:
https://docs.oracle.com/en/database/oracle/agent-memory/26.4/agmea/get-started.html#GUID-1DC2BEC9-4CAF-4668-BBBB-E9FC57C7E71E
4. Authentication setup for the 26AI database, including:
    a. Database username, password, and connection string (DSN)
    b. Wallet file for mTLS network configuration (if Mutual TLS (mTLS) authentication is set to Required in the database network settings)
    c. OCI config file (when using a local environment)


Tips: 
1. It is recommended to use a transactional database (ATP) rather than an analytical database (ADW) for faster response times.
2. For testing and R&D purposes, consider using the Developer database option, which is cost-effective and suitable for prototyping.
3. To avoid using a wallet file, you can add your IP address to the Access Control List (ACL) under the database network settings and set Mutual TLS (mTLS) authentication to Not Required.

## Importing Packages and Loading Environment Variables

In [26]:
from oracleagentmemory.apis.searchscope import SearchScope
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.dbschemapolicy import SchemaPolicy
from oracleagentmemory.core.embedders.embedder import Embedder
from oracleagentmemory.core.llms.llm import Llm
from oracleagentmemory.apis.searchscope import SearchScope

import oracledb
from oci.config import from_file

from dotenv import load_dotenv
import os
from pathlib import Path


In [3]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
CONNECT_STRING = os.getenv("DB_CONNECT_STRING")
COMPARTMENT_ID=os.getenv("COMPARTMENT_ID")

config = from_file(str(Path.home() / ".oci" / "config"))
oci_key_file = Path(config["key_file"]).expanduser()


## Connecting to the Database

In [4]:
def create_db_pool():
    return oracledb.create_pool(
        user=DB_USER,
        password=DB_PASSWORD,
        dsn=CONNECT_STRING,
    )


db_pool = create_db_pool()

try:
    with db_pool.acquire() as conn:
        with conn.cursor() as cursor:
            cursor.execute("SELECT 1 FROM DUAL")
            result = cursor.fetchone()
            print(f"Connection successful! Result: {result[0]}")
except Exception as err:
    print(f"Connection failed: {err}")

Connection successful! Result: 1


## OAM Setup

In [5]:
oci_embedder = Embedder(model="oci/cohere.embed-english-v3.0", 
      oci_compartment_id=COMPARTMENT_ID,
      oci_region=config['region'],
      oci_user=config['user'],
      oci_fingerprint=config['fingerprint'], 
      oci_tenancy=config['tenancy'],
      oci_key_file=oci_key_file)

oci_llm = Llm(model="oci/google.gemini-2.5-flash", # It is recommended to use a chat-capable model. When using text-only models (e.g., oci/openai.gpt-oss-120b), set extract_memories=False
     oci_compartment_id=COMPARTMENT_ID,
     oci_region=config['region'],
     oci_user=config['user'],
     oci_fingerprint=config['fingerprint'],
     oci_tenancy=config['tenancy'],
     oci_key_file=oci_key_file)

In [7]:
try:
 memory = OracleAgentMemory(connection=db_pool,
          embedder=oci_embedder,
          llm=oci_llm,
          schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
          table_name_prefix="OAM_")
#          extract_memories=False)   # Use this to prevent the LLM from extracting memories from messages when using text-only models. This setting can be overridden at the thread level.
except Exception as e:
 raise(e)

## Thread Creation and Message Management

In [8]:
# The thread is uniquely identified by thread_id. Including user_id provides better context and scoping.
thread = memory.create_thread(
    thread_id="05042026_1",
    user_id="assaf"
)

# Notes:
# 1. You can override the default extract_memories setting here (e.g., extract_memories=False).
# 2. For agent-based applications, consider adding agent_id so the thread is associated with both the user and the agent interacting with the user.

In [ ]:
messages = [
    {
        "role": "user",
        "content": (
            "is there any feature in Oracle db enable to organize agent's memeory"
            "Is this feature free? does it have an sdk?"
        ),
        # optionaly user can add message's id, timestamp, and metadata
    },
    {
        "role": "assistant",
        "content": (
            "Yes, there is a new feature - Oracle AI Agent Memory, which can store and organize enterprize data in the Oracle converged database."
            "Yes, it is free freature in the database, and you can utilize its SDK to integrate it in your agentic applications."
        ),
    },
]

thread.add_messages(messages)
# await thread.add_messages_async(messages) # to avoid async warning in notebooks

In [10]:
default_messages = thread.get_messages()
print([message.content for message in default_messages])

["is there any feature in Oracle db enable to organize agent's memeoryIs this feature free? does it have an sdk?", 'Yes, there is a new feature - Oracle AI Agent Memory, which can store and organize enterprize data in the Oracle converged database.Yes, it is free freature in the database, and you can utilize its SDK to integrate it in your agentic applications.']


In [13]:
default_messages

[Message(role='user', content="is there any feature in Oracle db enable to organize agent's memeoryIs this feature free? does it have an sdk?", timestamp='2026-05-04T06:51:59.643516', metadata=None, id='9f0a3a70-ac92-4e77-8e96-c53805b4f7f5'),
 Message(role='assistant', content='Yes, there is a new feature - Oracle AI Agent Memory, which can store and organize enterprize data in the Oracle converged database.Yes, it is free freature in the database, and you can utilize its SDK to integrate it in your agentic applications.', timestamp='2026-05-04T06:51:59.643516', metadata=None, id='80fdade9-1d4e-4724-8a74-0df2ca1edfed')]

In [ ]:
# The context card contains the essential information required for agent interaction.
# It includes facts, guidelines, preferences, and memories.
# It also includes recent messages and a summary.
context_card = thread.get_context_card()
print(context_card)

<context_card>
  <topics>
    <topic>oracle ai features</topic>
    <topic>ai agent memory</topic>
    <topic>sdk for agent memory</topic>
    <topic>free database features</topic>
    <topic>oracle database</topic>
    <topic>agent memory management</topic>
  </topics>
  <summary>
    A user inquired about a feature in Oracle Database for organizing agent memory. The assistant confirmed the existence of "Oracle AI Agent Memory," stating it's a free feature within the database and offers an SDK for integration into agentic applications.
  </summary>
  <relevant_information>
    <fact>
      <timestamp>2026-05-04T06:51:59.844826</timestamp>
      <content>Oracle AI Agent Memory is a free feature within the Oracle Database.</content>
    </fact>
    <fact>
      <timestamp>2026-05-04T06:51:59.844826</timestamp>
      <content>The Oracle AI Agent Memory feature is designed to store and organize enterprise data within the Oracle converged database.</content>
    </fact>
    <fact>
      <t

In [20]:
# you can also get the summary directly
summary = thread.get_summary() # users can also not include part of the history: except_last=1
print(summary)

A user inquired about a feature in Oracle DB for organizing agent memory. The assistant confirmed the existence of "Oracle AI Agent Memory," stating it's a free feature within the Oracle converged database for storing and organizing enterprise data, and that an SDK is available for integration.


In [22]:
# In some cases, it is useful to reopen the entire thread for further iteration.
previous_thread = memory.get_thread("05042026_1")
print(previous_thread.thread_id)

05042026_1


## Adding Memory Manually 

* Users can add memory manually, instead of relying on automatic extraction by the LLM from messages.
* Memory can be added with flexible scoping, such as user-level or thread-level.

In [24]:
# For thread specific memory addition just use the thread object
thread.add_memory("Oracle database became GA on March 2026.")

'7adcb705-78d6-4b7f-bb5b-bd9a026e1645'

In [25]:
# Global memory (applies to all threads for this user)
memory_id = memory.add_memory(
    "The user prefers short, bullet-point answers.",
    user_id="user_123",
    # You can also provide thread_id to scope the memory to a specific thread.
    # This can be useful if you want to add thread-scoped memory without reopening the thread.
)
print(memory_id)

9fe7947e-49d9-4de9-9224-1aeb64a121eb


## Search

* User can implement a semantic search in thread level or in a broader scope

In [ ]:
# For thread scope search use the thread object
results = thread.search("GA", max_results=2)
print([result.content for result in results])

In [29]:
# You can also focus on a sepcific memory type:
memory_results = thread.search("GA", max_results=2, record_types=["memory"])
print([result.content for result in memory_results])

['Oracle database became GA on March 2026.', 'Oracle database was made GA on March 2026.']


In [31]:
# Search within the user scope (across all threads for this user)
results_user_scope = memory.search(
    query="Is there a new feature in Oracle Database?",
    scope=SearchScope(user_id="assaf")  # You can also limit the search to a specific thread
)
for result in results_user_scope:
    print(f"- [{result.record.record_type}] {result.content}")

- [memory] Oracle database has a new feature, memeory agent.
- [memory] Oracle database has a new feature, memeory agent.
- [memory] Oracle database has a new feature, memeory agent.
- [memory] Oracle database has a new feature, memeory agent.
- [memory] Oracle database has a new feature, memeory agent.
- [memory] Oracle database has a new feature, memeory agent.
- [memory] Oracle database has a new feature, private container.
- [memory] Oracle database has a new feature, private container.
- [memory] Oracle database has a new feature, private container.
- [memory] Oracle database has a new feature, private container.


## Using Memory with the LLM

In [ ]:
memory_context = "\n".join(f"- {r.content}" for r in results[:3])

prompt = f"""
Use the memory below if relevant.

Memory:
{memory_context}

Question: are there new features in oracle db?
Answer:
"""

answer = oci_llm.generate(prompt) # using the model we defined earlier
print(answer.text)

According to the memory, Oracle database is set to become Generally Available (GA) in March 2026. This implies that it is not yet widely released.

Therefore, the question about "new features" doesn't apply in the traditional sense of updates to an existing, widely available product. The features that will be present are those of its initial GA release.
